# BiologicalProcess → Gene Relation Pipeline

Builds a unified, deduplicated edge table for the **BiologicalProcess–Gene** relation  
by ingesting processed files from multiple KG sources, normalising gene identifiers  
via NCBI/Ensembl mapping dictionaries, and writing the final triple table to disk.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**

- Pheknowlator
- PrimeKG


**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Mapping databases: `/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [5]:
import pandas as pd
import os

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_GENE/ALL_BIOLOGICALPROCESS_GENE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1 · Load Gene Identifier Mapping Dictionaries

Two mapping resources are loaded:
1. **Ensembl → NCBI Gene Symbol** (`ENS_2NCBI`): used to attach Ensembl IDs to NCBI symbols.
2. **NCBI gene_info** (`Homo_sapiens.gene_info`): used to resolve GeneID ↔ Symbol ↔ Description.

From these, lookup dictionaries are built for downstream normalisation:

In [6]:
# ── Ensembl ↔ NCBI Gene Symbol ────────────────────────────────────────────────
ens2ncbi = pd.read_csv(MAPPING_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ens2ncbi = ens2ncbi[~ens2ncbi['name'].isna()]          # drop rows without a gene symbol
NCBI_2ENS_dict = dict(zip(ens2ncbi['name'], ens2ncbi['initial_alias']))
del ens2ncbi

# ── NCBI gene_info (human) ────────────────────────────────────────────────────
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
ncbi['ENSEMBLE_ID'] = ncbi['Symbol'].map(NCBI_2ENS_dict)

NCBI_ID2sym_dict  = dict(zip(ncbi['GeneID'],   ncbi['Symbol']))
NCBI_ID2desc_dict = dict(zip(ncbi['GeneID'],   ncbi['description']))
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))

# Explode pipe-separated synonyms into individual keys → canonical symbol
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol

print(f"NCBI genes loaded: {len(ncbi):,}")
print(f"Synonym entries:   {len(NCBI_syn2sym_dict):,}")

NCBI genes loaded: 193,505
Synonym entries:   69,564


## 2 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and IDs configured.

### 1. Pheknowlator

In [7]:
Pheknowlator_BP_GENE = pd.read_csv(PROC_DIR + 'pheknowlator/BiologicalProcess_Gene.csv')
Pheknowlator_BP_GENE.columns = Pheknowlator_BP_GENE.columns.str.lower()

Pheknowlator_BP_GENE['tail_id_is'] = 'NCBI_ID'
Pheknowlator_BP_GENE['head_id_is'] = 'Quick_GO'
Pheknowlator_BP_GENE['relation'] = 'BiologicalProcess_Gene'
Pheknowlator_BP_GENE['head_type'] = 'BiologicalProcess'
Pheknowlator_BP_GENE['tail_type'] = 'Gene'
Pheknowlator_BP_GENE['kg_source'] = 'pheknowlator'
Pheknowlator_BP_GENE['kg_type'] = 'Generalised'
Pheknowlator_BP_GENE['species'] = 'Homo sapiens'
print(f"Pheknowlator BP→Gene: {len(Pheknowlator_BP_GENE):,} rows")
Pheknowlator_BP_GENE

Pheknowlator BP→Gene: 278,928 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,kg_source,tail_id_is,kg_type,species
0,GO:0071222,BiologicalProcess_Gene,PF4V1,BiologicalProcess,Gene,cellular response to lipopolysaccharide,platelet factor 4 variant 1,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
1,GO:0008360,BiologicalProcess_Gene,ARHGAP15,BiologicalProcess,Gene,regulation of cell shape,Rho GTPase activating protein 15,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
2,GO:0006915,BiologicalProcess_Gene,BCL2L14,BiologicalProcess,Gene,apoptotic process,BCL2 like 14,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
3,GO:0072659,BiologicalProcess_Gene,RAMP2,BiologicalProcess,Gene,protein localization to plasma membrane,receptor activity modifying protein 2,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
4,GO:0000070,BiologicalProcess_Gene,NDC80,BiologicalProcess,Gene,mitotic sister chromatid segregation,NDC80 kinetochore complex component,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...
278923,GO:1903409,BiologicalProcess_Gene,INAVA,BiologicalProcess,Gene,reactive oxygen species biosynthetic process,innate immunity activator,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
278924,GO:0007165,BiologicalProcess_Gene,HNRNPK,BiologicalProcess,Gene,signal transduction,heterogeneous nuclear ribonucleoprotein K,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
278925,GO:0006977,BiologicalProcess_Gene,CASP2,BiologicalProcess,Gene,"obsolete DNA damage response, signal transduct...",caspase 2,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens
278926,GO:0002407,BiologicalProcess_Gene,TRPM4,BiologicalProcess,Gene,dendritic cell chemotaxis,transient receptor potential cation channel su...,Quick_GO,pheknowlator,NCBI_ID,Generalised,Homo sapiens


### 2. PrimeKG

In [10]:
PrimeKG_BP_GENE = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_BiologicalProcess_Gene.csv')
PrimeKG_BP_GENE.columns = PrimeKG_BP_GENE.columns.str.lower()

PrimeKG_BP_GENE['tail_id_is'] = 'NCBI_ID'
PrimeKG_BP_GENE['head_id_is'] = 'Quick_GO'
PrimeKG_BP_GENE['kg_source'] = 'PrimeKG'
PrimeKG_BP_GENE['kg_type'] = 'Generalised'
PrimeKG_BP_GENE['species'] = 'Homo sapiens'

print(f"PrimeKG BP→Gene: {len(PrimeKG_BP_GENE):,} rows")
PrimeKG_BP_GENE

PrimeKG BP→Gene: 144,805 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,GO:0043312,BiologicalProcess_Gene,A1BG,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,neutrophil degranulation,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised,Homo sapiens
1,GO:0043312,BiologicalProcess_Gene,SERPINA3,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,neutrophil degranulation,SERPINA3,serpin family A member 3,ENSG00000196136,Generalised,Homo sapiens
2,GO:0043312,BiologicalProcess_Gene,AOC1,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,neutrophil degranulation,AOC1,amine oxidase copper containing 1,ENSG00000002726,Generalised,Homo sapiens
3,GO:0043312,BiologicalProcess_Gene,ACAA1,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,neutrophil degranulation,ACAA1,acetyl-CoA acyltransferase 1,ENSG00000060971,Generalised,Homo sapiens
4,GO:0043312,BiologicalProcess_Gene,ACLY,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,neutrophil degranulation,ACLY,ATP citrate lyase,ENSG00000131473,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144800,GO:0032597,BiologicalProcess_Gene,CD24,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,B cell receptor transport into membrane raft,CD24,CD24 molecule,ENSG00000272398,Generalised,Homo sapiens
144801,GO:0032600,BiologicalProcess_Gene,CD24,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,chemokine receptor transport out of membrane raft,CD24,CD24 molecule,ENSG00000272398,Generalised,Homo sapiens
144802,GO:0051494,BiologicalProcess_Gene,IQCJ-SCHIP1,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,negative regulation of cytoskeleton organization,IQCJ-SCHIP1,IQCJ-SCHIP1 readthrough,ENSG00000283154,Generalised,Homo sapiens
144803,GO:0090133,BiologicalProcess_Gene,APELA,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,mesendoderm migration,APELA,apelin receptor early endogenous ligand,ENSG00000248329,Generalised,Homo sapiens


## 3 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [12]:
source_dfs = [
    Pheknowlator_BP_GENE,
    PrimeKG_BP_GENE
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df['species'] = 'Homo sapiens'
final_df

Consolidated rows: 423,733


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0071222,BiologicalProcess_Gene,PF4V1,BiologicalProcess,None,Gene,pheknowlator,Quick_GO,NCBI_ID,cellular response to lipopolysaccharide,platelet factor 4 variant 1,Generalised,Homo sapiens
1,GO:0008360,BiologicalProcess_Gene,ARHGAP15,BiologicalProcess,None,Gene,pheknowlator,Quick_GO,NCBI_ID,regulation of cell shape,Rho GTPase activating protein 15,Generalised,Homo sapiens
2,GO:0006915,BiologicalProcess_Gene,BCL2L14,BiologicalProcess,None,Gene,pheknowlator,Quick_GO,NCBI_ID,apoptotic process,BCL2 like 14,Generalised,Homo sapiens
3,GO:0072659,BiologicalProcess_Gene,RAMP2,BiologicalProcess,None,Gene,pheknowlator,Quick_GO,NCBI_ID,protein localization to plasma membrane,receptor activity modifying protein 2,Generalised,Homo sapiens
4,GO:0000070,BiologicalProcess_Gene,NDC80,BiologicalProcess,None,Gene,pheknowlator,Quick_GO,NCBI_ID,mitotic sister chromatid segregation,NDC80 kinetochore complex component,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
423728,GO:0032597,BiologicalProcess_Gene,CD24,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,B cell receptor transport into membrane raft,CD24 molecule,Generalised,Homo sapiens
423729,GO:0032600,BiologicalProcess_Gene,CD24,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,chemokine receptor transport out of membrane raft,CD24 molecule,Generalised,Homo sapiens
423730,GO:0051494,BiologicalProcess_Gene,IQCJ-SCHIP1,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,negative regulation of cytoskeleton organization,IQCJ-SCHIP1 readthrough,Generalised,Homo sapiens
423731,GO:0090133,BiologicalProcess_Gene,APELA,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,mesendoderm migration,apelin receptor early endogenous ligand,Generalised,Homo sapiens


## 4 · Sanity Check — Distinct Values

In [14]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'kg_type', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'BiologicalProcess_Gene'}
head_type           : {'BiologicalProcess'}
tail_type           : {'Gene'}
relation_type       : {None, 'interacts with'}
kg_source           : {'PrimeKG', 'pheknowlator'}
kg_type             : {'Generalised'}
head_id_is          : {'Quick_GO'}
tail_id_is          : {'NCBI_ID'}


## 5 · Gene Name Normalisation (tail)

Rows where `tail_detail_name` is missing are resolved using:
1. Strip hyphens from the tail gene symbol.
2. Map through the synonym dictionary to get the canonical NCBI symbol.
3. Map canonical symbol to its description via `NCBI_sym2desc_dict`.
4. Drop rows where `tail_detail_name` remains unresolved.

In [15]:
# Step 1-2: resolve synonyms for rows with missing tail_detail_name
mask = final_df['tail_detail_name'].isna()
final_df.loc[mask, 'tail'] = (
    final_df.loc[mask, 'tail']
    .str.replace('-', '', regex=False)
    .map(NCBI_syn2sym_dict)
    .fillna(final_df.loc[mask, 'tail'])
)

# Step 3: fill description for still-missing rows
mask = final_df['tail_detail_name'].isna()
final_df.loc[mask, 'tail_detail_name'] = final_df.loc[mask, 'tail'].map(NCBI_sym2desc_dict)

# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 56 unresolvable rows → 423,677 remaining


## 6 · NaN Audit (pre-dedup)

In [16]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,278874,0,278874
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 7 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [17]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 423,677  |  After dedup: 155,018


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,GO:0000002,BiologicalProcess_Gene,AKT3,BiologicalProcess,interacts with,Gene,PrimeKG::pheknowlator,Quick_GO,NCBI_ID,mitochondrial genome maintenance,AKT serine/threonine kinase 3
1,GO:0000002,BiologicalProcess_Gene,LONP1,BiologicalProcess,interacts with,Gene,PrimeKG::pheknowlator,Quick_GO,NCBI_ID,mitochondrial genome maintenance,"lon peptidase 1, mitochondrial"
2,GO:0000002,BiologicalProcess_Gene,MED12,BiologicalProcess,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,mitochondrial genome maintenance,mediator complex subunit 12


## 8 · Post-dedup NaN Audit & Source Distribution

In [18]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

display(pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
}))

print("\nkg_source values present:", set(final_df_dedup['kg_source']))

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,11511,0,11511
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0



kg_source values present: {'PrimeKG::pheknowlator', 'pheknowlator', 'PrimeKG'}


In [19]:
# final_df_dedup

## 10 · Save Output

In [21]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 155,018 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_GENE/ALL_BIOLOGICALPROCESS_GENE.csv


In [23]:
# OUT_PATH